# LeetCode 1008 — Construct Binary Search Tree from Preorder Traversal bằng Python

Link đề: https://leetcode.com/problems/construct-binary-search-tree-from-preorder-traversal/ · Độ khó: **Medium** · Chủ đề: Array, Stack, Tree, Binary Search Tree, Binary Tree, Monotonic Stack

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def bstFromPreorder(self, preorder: List[int]) -> Optional[TreeNode]:
        ...
```

Ghi chú: đáp án là **một cây duy nhất** (các giá trị đôi một khác nhau nên từ duyệt tiền thứ tự của BST ta khôi phục được đúng một cây — mục 10 chứng minh, mục 12 vét cạn). Vì LeetCode trả về nút gốc, notebook **tự dựng** `TreeNode`, hàm dựng cây từ danh sách kiểu LeetCode (`build_tree`, theo tầng, `None` là nút trống) và hàm xuất `to_level` để so sánh với đầu ra mẫu `[8,5,10,1,7,null,12]`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho duyệt **tiền thứ tự** (nút, trái, phải) của một cây nhị phân tìm kiếm (BST), dựng lại cây.
2. Nắm công cụ: tính chất BST (trái `<` nút `<` phải), tiền thứ tự = nút đầu rồi **khối nhỏ hơn** rồi **khối lớn hơn**, đệ quy với **cận trên**, ngăn xếp đơn điệu, tìm nhị phân điểm chia.
3. Đi từ **chèn lần lượt từng giá trị vào BST** → **tách mảng tại phần tử đầu tiên lớn hơn gốc** → **đệ quy với cận trên (một con trỏ dùng chung)** → **ngăn xếp** → **tách bằng `bisect`**.
4. Hiểu mấu chốt: sau gốc, preorder gồm một **khối toàn nhỏ hơn gốc** (cây con trái) rồi một **khối toàn lớn hơn gốc** (cây con phải); các điểm chia giúp dựng bằng `O(n)`.
5. Tránh lỗi kinh điển: so sánh mỗi giá trị chỉ với nút ngay trước (bỏ qua tổ tiên), thiếu cận trên khi đệ quy, con trỏ chỉ số cục bộ, tách lệch một phần tử, không `pop` hết các nút nhỏ hơn trong ngăn xếp, dựng cây cân bằng từ mảng đã sắp (đúng BST nhưng **sai** tiền thứ tự).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `preorder` là duyệt **tiền thứ tự** của một cây nhị phân tìm kiếm (BST). Dựng cây và trả về nút gốc. Luôn tồn tại BST thoả điều kiện cho dữ liệu kiểm thử.

BST: với mọi nút, mọi nút trong cây con trái có giá trị **nhỏ hơn hẳn**, mọi nút trong cây con phải có giá trị **lớn hơn hẳn**. Tiền thứ tự: nút trước, rồi cây con trái, rồi cây con phải.

**Ví dụ 1:** `preorder = [8,5,1,7,10,12]` → `[8,5,10,1,7,null,12]`.

**Ví dụ 2:** `preorder = [1,3]` → `[1,null,3]`.

**Ràng buộc:** `1 <= preorder.length <= 100`; `1 <= preorder[i] <= 1000`; mọi giá trị trong `preorder` **khác nhau**.

In [1]:
import bisect
import itertools
import math
import random
import sys
import time
from collections import deque
from typing import List, Optional

sys.setrecursionlimit(5000)


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"


def build_tree(values):
    """Dựng cây từ danh sách theo tầng kiểu LeetCode (None = không có nút)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def to_level(root):
    """Xuất cây theo tầng kiểu LeetCode (bỏ None ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


def tup(root):
    return None if root is None else (root.val, tup(root.left), tup(root.right))


def from_tuple(t):
    return None if t is None else TreeNode(t[0], from_tuple(t[1]), from_tuple(t[2]))


def preorder_of(root):
    out = []
    stack = [root]
    while stack:
        x = stack.pop()
        if x:
            out.append(x.val)
            stack.append(x.right)
            stack.append(x.left)
    return out


def inorder_of(root):
    out, stack, x = [], [], root
    while stack or x:
        while x:
            stack.append(x)
            x = x.left
        x = stack.pop()
        out.append(x.val)
        x = x.right
    return out


def size(root):
    return len(preorder_of(root))


ex = [([8, 5, 1, 7, 10, 12], [8, 5, 10, 1, 7, None, 12]), ([1, 3], [1, None, 3])]
t = build_tree(ex[0][1])
print(tup(t), to_level(t), preorder_of(t), inorder_of(t))
assert preorder_of(t) == ex[0][0] and inorder_of(t) == sorted(ex[0][0])

(8, (5, (1, None, None), (7, None, None)), (10, None, (12, None, None))) [8, 5, 10, 1, 7, None, 12] [8, 5, 1, 7, 10, 12] [1, 5, 7, 8, 10, 12]


## 3. Kiến thức nền

### 3.1 Tính chất BST và duyệt trung thứ tự

Duyệt **trung thứ tự** (trái, nút, phải) của BST cho dãy **tăng dần**. Do đó với BST có cùng tập giá trị, `sorted(preorder)` chính là trung thứ tự.

### 3.2 Cấu trúc của preorder BST

`preorder = [root] + [các giá trị cây con trái] + [các giá trị cây con phải]`. Vì mọi giá trị ở cây trái `< root < ` mọi giá trị ở cây phải, nên sau `root` là một **khối nhỏ hơn root** rồi một **khối lớn hơn root** (mỗi khối có thể rỗng). Điểm chia = vị trí đầu tiên có giá trị `> root`.

In [2]:
pre = [8, 5, 1, 7, 10, 12]
k = next((i for i in range(1, len(pre)) if pre[i] > pre[0]), len(pre))
print("root =", pre[0], "| khối trái:", pre[1:k], "| khối phải:", pre[k:])
assert all(x < pre[0] for x in pre[1:k]) and all(x > pre[0] for x in pre[k:])

root = 8 | khối trái: [5, 1, 7] | khối phải: [10, 12]


### 3.3 Hàm kiểm tra tính hợp lệ

`is_bst(root)` kiểm tra BST bằng cận `(lo, hi)`; `valid_answer(root, preorder)` kiểm: đúng là BST, tiền thứ tự bằng `preorder`, cùng số nút. Vì giá trị khác nhau, cây thoả ba điều kiện này là **duy nhất** (mục 10).

In [3]:
def is_bst(root, lo=float("-inf"), hi=float("inf")):
    stack = [(root, lo, hi)]
    while stack:
        node, a, b = stack.pop()
        if node is None:
            continue
        if not (a < node.val < b):
            return False
        stack.append((node.left, a, node.val))
        stack.append((node.right, node.val, b))
    return True


def valid_answer(root, preorder):
    return is_bst(root) and preorder_of(root) == list(preorder)


print(valid_answer(build_tree([8, 5, 10, 1, 7, None, 12]), [8, 5, 1, 7, 10, 12]), valid_answer(build_tree([8, 5, 10, 1, 7, None, 12]), [8, 5, 1, 7, 12, 10]))

True False


## 4. Ý tưởng nền tảng / trực giác

`preorder = [8, 5, 1, 7, 10, 12]`. Gốc `8`. Các giá trị tiếp theo `5, 1, 7` đều `< 8` ⇒ cây con trái; `10, 12` đều `> 8` ⇒ cây con phải. Đệ quy:

- trái `[5, 1, 7]`: gốc `5`; `1` (`< 5`) trái; `7` (`> 5`) phải;
- phải `[10, 12]`: gốc `10`; `12` phải.

Kết quả `8(5(1,7), 10(·,12))` = `[8,5,10,1,7,null,12]`. Thay vì tách mảng (`O(n²)` xấu nhất), ta có thể duyệt một lượt với **cận trên**: dựng cây con trái trong khi giá trị kế tiếp `< root`.

In [4]:
def show_split(pre, depth=0):
    if not pre:
        return
    k = next((i for i in range(1, len(pre)) if pre[i] > pre[0]), len(pre))
    print("  " * depth + f"gốc {pre[0]}: trái {pre[1:k]} | phải {pre[k:]}")
    show_split(pre[1:k], depth + 1)
    show_split(pre[k:], depth + 1)


show_split([8, 5, 1, 7, 10, 12])

gốc 8: trái [5, 1, 7] | phải [10, 12]
  gốc 5: trái [1] | phải [7]
    gốc 1: trái [] | phải []
    gốc 7: trái [] | phải []
  gốc 10: trái [] | phải [12]
    gốc 12: trái [] | phải []


## 5. Brute force — chèn lần lượt vào BST

**Ý tưởng:** bắt đầu với `root = TreeNode(preorder[0])`; với mỗi giá trị tiếp theo, đi từ gốc xuống: nếu nhỏ hơn nút hiện tại thì sang trái, ngược lại sang phải, tới chỗ trống thì gắn. Mỗi lần chèn `O(h)` ⇒ `O(n·h)`, xấu nhất `O(n²)` (cây lệch). Cách này **đúng** vì chèn theo thứ tự tiền thứ tự tái tạo đúng cấu trúc (mục 10), nên cũng là một **oracle** độc lập với các cách tách mảng.

In [5]:
def bst_insert(preorder):
    root = TreeNode(preorder[0])
    for v in preorder[1:]:
        cur = root
        while True:
            if v < cur.val:
                if cur.left is None:
                    cur.left = TreeNode(v)
                    break
                cur = cur.left
            else:
                if cur.right is None:
                    cur.right = TreeNode(v)
                    break
                cur = cur.right
    return root


for pre, want in ex:
    r = bst_insert(pre)
    assert to_level(r) == want and valid_answer(r, pre)
print([to_level(bst_insert(p)) for p, _ in ex])

[[8, 5, 10, 1, 7, None, 12], [1, None, 3]]


In [6]:
# Code nộp LeetCode cho Brute force (chèn lần lượt)
class SolutionInsert:
    def bstFromPreorder(self, preorder: List[int]) -> Optional[TreeNode]:
        root = TreeNode(preorder[0])
        for v in preorder[1:]:
            cur = root
            while True:
                if v < cur.val:
                    if cur.left is None:
                        cur.left = TreeNode(v)
                        break
                    cur = cur.left
                else:
                    if cur.right is None:
                        cur.right = TreeNode(v)
                        break
                    cur = cur.right
        return root

**Độ phức tạp:** `O(n·h)` thời gian (≤ `O(n²)`), `O(h)` bộ nhớ phụ.

**Nút thắt:** mỗi giá trị đi lại từ gốc, dù ta biết nó nằm ngay dưới giá trị trước hoặc ngay dưới một tổ tiên gần.

## 6. Bước đột phá

Preorder = gốc, khối nhỏ hơn, khối lớn hơn. Dùng **cận trên** `bound`: `build(bound)` dựng cây con từ vị trí hiện tại với mọi giá trị `< bound`; dừng khi giá trị kế tiếp `≥ bound`. Cây con trái của `root` có cận `root.val`; cây con phải kế thừa cận của `root`. Mỗi giá trị được tiêu thụ đúng một lần ⇒ `O(n)`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** tách mảng tại phần tử đầu tiên lớn hơn gốc, **Cách 3** đệ quy với cận trên (đáp án chính), **Cách 4** ngăn xếp, **Cách 5** tách bằng `bisect` trên vị vị từ đơn điệu.

### Cách 2 — Tách mảng tại phần tử đầu tiên lớn hơn gốc

**Giải thích:** `root = pre[0]`; `k` = chỉ số đầu tiên (từ `1`) có `pre[k] > root` (hoặc `len(pre)`); cây trái dựng từ `pre[1:k]`, cây phải từ `pre[k:]`.

In [7]:
def bst_split(pre):
    if not pre:
        return None
    k = 1
    while k < len(pre) and pre[k] < pre[0]:
        k += 1                                          # k: đầu khối "lớn hơn gốc"
    return TreeNode(pre[0], bst_split(pre[1:k]), bst_split(pre[k:]))


for pre, want in ex:
    assert to_level(bst_split(pre)) == want
print([to_level(bst_split(p)) for p, _ in ex])

[[8, 5, 10, 1, 7, None, 12], [1, None, 3]]


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionSplit:
    def bstFromPreorder(self, preorder: List[int]) -> Optional[TreeNode]:
        if not preorder:
            return None
        k = 1
        while k < len(preorder) and preorder[k] < preorder[0]:
            k += 1
        return TreeNode(preorder[0], self.bstFromPreorder(preorder[1:k]), self.bstFromPreorder(preorder[k:]))

**Độ phức tạp:** `O(n²)` xấu nhất (cắt mảng và tìm `k` lặp lại trên cây lệch), `O(n)` ngăn xếp; với `n ≤ 100` thoải mái.

### Cách 3 — Đệ quy với cận trên (con trỏ dùng chung)

**Ý tưởng:** `idx` là con trỏ dùng chung vào `preorder`. `build(bound)`: nếu `idx` hết hoặc `preorder[idx] > bound` ⇒ `None`; ngược lại tạo nút `preorder[idx]`, `idx += 1`, `left = build(node.val)`, `right = build(bound)`.

In [9]:
def bst_bound(preorder):
    idx = 0
    n = len(preorder)

    def build(bound):
        nonlocal idx
        if idx == n or preorder[idx] > bound:
            return None
        node = TreeNode(preorder[idx])
        idx += 1
        node.left = build(node.val)                      # trái: giá trị < node.val
        node.right = build(bound)                        # phải: giá trị < bound (cận của tổ tiên)
        return node

    return build(float("inf"))


for pre, want in ex:
    assert to_level(bst_bound(pre)) == want
print([to_level(bst_bound(p)) for p, _ in ex])

[[8, 5, 10, 1, 7, None, 12], [1, None, 3]]


In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionBound:
    def bstFromPreorder(self, preorder: List[int]) -> Optional[TreeNode]:
        idx, n = 0, len(preorder)

        def build(bound):
            nonlocal idx
            if idx == n or preorder[idx] > bound:
                return None
            node = TreeNode(preorder[idx])
            idx += 1
            node.left = build(node.val)
            node.right = build(bound)
            return node

        return build(float("inf"))

**Độ phức tạp:** `O(n)` thời gian (mỗi giá trị tiêu thụ đúng một lần), `O(h)` ngăn xếp.

### Cách 4 — Ngăn xếp đơn điệu

**Ý tưởng:** ngăn xếp chứa đường đi từ gốc tới nút vừa chèn (giá trị **giảm dần** từ đáy tới đỉnh khi đi trái). Với giá trị mới `v`: bỏ (pop) mọi nút có giá trị `< v`, nút cuối cùng bị pop là cha mà `v` là con **phải**; nếu không pop gì thì `v` là con **trái** của đỉnh. Rồi đẩy `v`.

In [11]:
def bst_stack(preorder):
    root = TreeNode(preorder[0])
    stack = [root]
    for v in preorder[1:]:
        node = TreeNode(v)
        last = None
        while stack and stack[-1].val < v:
            last = stack.pop()                           # cha nhỏ nhất vẫn nhỏ hơn v
        if last is not None:
            last.right = node                            # v là con phải của nút bị pop cuối cùng
        else:
            stack[-1].left = node                        # không pop: v nhỏ hơn đỉnh => con trái
        stack.append(node)
    return root


for pre, want in ex:
    assert to_level(bst_stack(pre)) == want
print([to_level(bst_stack(p)) for p, _ in ex])

[[8, 5, 10, 1, 7, None, 12], [1, None, 3]]


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionStack:
    def bstFromPreorder(self, preorder: List[int]) -> Optional[TreeNode]:
        root = TreeNode(preorder[0])
        stack = [root]
        for v in preorder[1:]:
            node = TreeNode(v)
            last = None
            while stack and stack[-1].val < v:
                last = stack.pop()
            if last:
                last.right = node
            else:
                stack[-1].left = node
            stack.append(node)
        return root

**Độ phức tạp:** `O(n)` thời gian (mỗi nút vào/ra ngăn xếp một lần), `O(h)` bộ nhớ.

### Cách 5 — Tách bằng `bisect` trên vị từ đơn điệu

**Ý tưởng:** trong đoạn `pre[lo+1 : hi]` (sau gốc), dãy các giá trị là `[nhỏ hơn gốc ... ][lớn hơn gốc ...]` nên vị từ `pre[i] > root` là `False…False True…True` — **đơn điệu**, tìm điểm chia bằng nhị phân. Dùng `bisect_right` với khoá (Python 3.10+: tham số `key`): `bisect.bisect_right(pre, root, lo+1, hi, key=...)`? Ở đây vị từ không phải khoá đơn điệu theo giá trị nên ta tự viết nhị phân.

In [13]:
def bst_bisect(preorder):
    def build(lo, hi):                                   # dựng cây từ preorder[lo:hi]
        if lo >= hi:
            return None
        root = preorder[lo]
        a, b = lo + 1, hi                                # tìm k nhỏ nhất trong [lo+1, hi] với preorder[k] > root
        while a < b:
            mid = (a + b) // 2
            if preorder[mid] > root:
                b = mid
            else:
                a = mid + 1
        return TreeNode(root, build(lo + 1, a), build(a, hi))

    return build(0, len(preorder))


for pre, want in ex:
    assert to_level(bst_bisect(pre)) == want
print([to_level(bst_bisect(p)) for p, _ in ex])

[[8, 5, 10, 1, 7, None, 12], [1, None, 3]]


In [14]:
# Code nộp LeetCode cho Cách 5
class SolutionBisect:
    def bstFromPreorder(self, preorder: List[int]) -> Optional[TreeNode]:
        def build(lo, hi):
            if lo >= hi:
                return None
            root = preorder[lo]
            a, b = lo + 1, hi
            while a < b:
                mid = (a + b) // 2
                if preorder[mid] > root:
                    b = mid
                else:
                    a = mid + 1
            return TreeNode(root, build(lo + 1, a), build(a, hi))

        return build(0, len(preorder))

**Độ phức tạp:** `T(n) = T(k) + T(n − k − 1) + O(log n)` ⇒ `O(n log n)` thời gian; `O(h)` ngăn xếp (không cắt mảng nên không có chi phí sao chép).

## 8. TRACE từng bước

**Trace Cách 3 (cận trên)** với `preorder = [8, 5, 1, 7, 10, 12]`:

| gọi | `idx` | `preorder[idx]` | cận | kết quả |
|---|---|---|---|---|
| `build(∞)` | 0 | 8 | ∞ | tạo `8`, `idx = 1` |
| `build(8)` (trái của 8) | 1 | 5 | 8 | tạo `5`, `idx = 2` |
| `build(5)` (trái của 5) | 2 | 1 | 5 | tạo `1`, `idx = 3` |
| `build(1)` (trái của 1) | 3 | 7 | 1 | `7 > 1` ⇒ `None` |
| `build(5)` (phải của 1) | 3 | 7 | 5 | `7 > 5` ⇒ `None` |
| `build(5)` (phải của 5) | 3 | 7 | 8 | tạo `7`, `idx = 4` |
| … | 4 | 10 | … | con của `7` đều `None` (cận `7`, `8`); quay lên |
| `build(∞)` (phải của 8) | 4 | 10 | ∞ | tạo `10`, `idx = 5`; trái `None` (cận 10); phải tạo `12` |

Hàm trace in từng lời gọi:

In [15]:
def trace_bound(preorder):
    idx = 0
    n = len(preorder)

    def build(bound, depth):
        nonlocal idx
        pad = "  " * depth
        if idx == n:
            print(f"{pad}build(cận={bound}): hết dữ liệu -> None")
            return None
        if preorder[idx] > bound:
            print(f"{pad}build(cận={bound}): {preorder[idx]} > {bound} -> None")
            return None
        node = TreeNode(preorder[idx])
        print(f"{pad}build(cận={bound}): tạo {node.val} (idx {idx} -> {idx + 1})")
        idx += 1
        node.left = build(node.val, depth + 1)
        node.right = build(bound, depth + 1)
        return node

    root = build(float("inf"), 0)
    print("=>", to_level(root))
    return root


assert to_level(trace_bound([8, 5, 1, 7, 10, 12])) == [8, 5, 10, 1, 7, None, 12]

build(cận=inf): tạo 8 (idx 0 -> 1)
  build(cận=8): tạo 5 (idx 1 -> 2)
    build(cận=5): tạo 1 (idx 2 -> 3)
      build(cận=1): 7 > 1 -> None
      build(cận=5): 7 > 5 -> None
    build(cận=8): tạo 7 (idx 3 -> 4)
      build(cận=7): 10 > 7 -> None
      build(cận=8): 10 > 8 -> None
  build(cận=inf): tạo 10 (idx 4 -> 5)
    build(cận=10): 12 > 10 -> None
    build(cận=inf): tạo 12 (idx 5 -> 6)
      build(cận=12): hết dữ liệu -> None
      build(cận=inf): hết dữ liệu -> None
=> [8, 5, 10, 1, 7, None, 12]


**Trace Cách 4 (ngăn xếp)** với `[8, 5, 1, 7, 10, 12]`:

| `v` | pop | gắn | ngăn xếp sau |
|---|---|---|---|
| 5 | không | `5` là con trái của `8` | `[8, 5]` |
| 1 | không | `1` là con trái của `5` | `[8, 5, 1]` |
| 7 | `1`, `5` | `7` là con phải của `5` (pop cuối) | `[8, 7]` |
| 10 | `7`, `8` | `10` là con phải của `8` (pop cuối) | `[10]` |
| 12 | `10` | `12` là con phải của `10` | `[12]` |

Chú ý: với `v = 7` ta pop `1` (`1 < 7`) rồi pop `5` (`5 < 7`); đỉnh `8 > 7` nên dừng; nút bị pop **cuối cùng** là `5` ⇒ `7` là con phải của `5`.

In [16]:
def trace_stack(preorder):
    root = TreeNode(preorder[0])
    stack = [root]
    print("bắt đầu: ngăn xếp", [x.val for x in stack])
    for v in preorder[1:]:
        node = TreeNode(v)
        popped = []
        last = None
        while stack and stack[-1].val < v:
            last = stack.pop()
            popped.append(last.val)
        if last is not None:
            last.right = node
            where = f"con PHẢI của {last.val}"
        else:
            stack[-1].left = node
            where = f"con TRÁI của {stack[-1].val}"
        stack.append(node)
        print(f"  v={v}: pop {popped} -> {where}; ngăn xếp {[x.val for x in stack]}")
    print("=>", to_level(root))
    return root


assert to_level(trace_stack([8, 5, 1, 7, 10, 12])) == [8, 5, 10, 1, 7, None, 12]

bắt đầu: ngăn xếp [8]
  v=5: pop [] -> con TRÁI của 8; ngăn xếp [8, 5]
  v=1: pop [] -> con TRÁI của 5; ngăn xếp [8, 5, 1]
  v=7: pop [1, 5] -> con PHẢI của 5; ngăn xếp [8, 7]
  v=10: pop [7, 8] -> con PHẢI của 8; ngăn xếp [10]
  v=12: pop [10] -> con PHẢI của 10; ngăn xếp [12]
=> [8, 5, 10, 1, 7, None, 12]


## 9. Code nộp LeetCode

Chọn **Cách 3** (đệ quy với cận trên): `O(n)`, ngắn. Giữ nguyên `class Solution`, `bstFromPreorder(self, preorder)`:

In [17]:
class Solution:
    def bstFromPreorder(self, preorder: List[int]) -> Optional[TreeNode]:
        idx, n = 0, len(preorder)                       # con trỏ dùng chung vào preorder

        def build(bound):
            nonlocal idx
            if idx == n or preorder[idx] > bound:       # hết dữ liệu hoặc giá trị thuộc về tổ tiên
                return None
            node = TreeNode(preorder[idx])
            idx += 1
            node.left = build(node.val)                 # cây con trái: giá trị < node.val
            node.right = build(bound)                   # cây con phải: giá trị < cận của tổ tiên
            return node

        return build(float("inf"))


print([to_level(Solution().bstFromPreorder(p)) for p, _ in ex])
assert all(to_level(Solution().bstFromPreorder(p)) == w for p, w in ex)

[[8, 5, 10, 1, 7, None, 12], [1, None, 3]]


## 10. Vì sao đúng

**(a) Cây duy nhất.** Giả sử hai BST `T₁`, `T₂` (các giá trị khác nhau) có cùng preorder `P`. Quy nạp theo độ dài: gốc đều là `P[0]`. Cây con trái của mỗi cây gồm đúng các giá trị `< P[0]` trong `P` (tính chất BST), và cây con phải gồm các giá trị `> P[0]`; trong preorder, các giá trị trái đứng liền sau gốc rồi tới phải, nên cả hai cây có cùng tập và cùng thứ tự các giá trị trái/phải ⇒ cùng preorder con ⇒ theo quy nạp hai cây con trái bằng nhau, hai cây con phải bằng nhau. Vậy `T₁ = T₂`.

**(b) Cấu trúc preorder.** Preorder của BST `= [root] + pre(trái) + pre(phải)` với mọi giá trị trái `< root <` mọi giá trị phải. Do đó điểm chia là vị trí đầu tiên có giá trị `> root` (Cách 2) và vị từ `pre[i] > root` đơn điệu (Cách 5).

**(c) Cận trên (Cách 3).** Bất biến: lời gọi `build(bound)` bắt đầu tại `idx` và tiêu thụ **mọi giá trị liên tiếp `< bound`** (kể từ `idx`) tạo thành đúng một cây con BST. Với nút mới `v = preorder[idx]`: cây con trái của nó gồm các giá trị liên tiếp tiếp theo `< v` (`build(v)`); sau khi dừng ở giá trị `≥ v` (thực ra `> v`), cây con phải gồm các giá trị liên tiếp `> v` và `< bound` (`build(bound)` — các giá trị `> v` nhưng phải `< bound` để còn thuộc cây con này, không phải của tổ tiên). Dừng khi giá trị `> bound` (thuộc tổ tiên). Mỗi giá trị được dùng đúng một lần.

**(d) Ngăn xếp (Cách 4).** Ngăn xếp luôn là đường đi từ gốc tới nút vừa chèn theo các nút **chưa có cây con phải đóng**: tức là chuỗi các nút mà nút sau là con trái của nút trước. Với `v` mới: các nút trên ngăn xếp có giá trị `< v` không thể có thêm cây con trái chứa `v`; `v` phải nằm ở cây con phải của nút **sâu nhất** có giá trị `< v` mà cha của nó (tiếp theo trong ngăn xếp) có giá trị `> v` — chính là nút pop **cuối cùng**. Nếu không có nút `< v` thì `v` là con trái của đỉnh. Đẩy `v` lên ngăn xếp vì `v` giờ là nút cuối đường đi.

**(e) Chèn lần lượt (Cách 1).** Khi chèn các giá trị theo thứ tự preorder, mỗi giá trị đi theo đường BST và dừng ở đúng vị trí mà cấu trúc preorder đòi hỏi (cha của nó đã được chèn trước vì cha đứng trước con trong preorder); kết quả là BST có preorder đúng bằng dãy vào, nên theo (a) là cây duy nhất.

**(f) Kiểm chứng bằng máy.** Với mọi `n ≤ 8`: mọi BST `n` nút trên nhãn `1..n` (`Catalan(n)` cây) cho `Catalan(n)` preorder **phân biệt**, và mọi cách dựng lại **đúng** cây gốc từ preorder của nó; trong `n!` hoán vị của `1..n` có đúng `Catalan(n)` hoán vị là preorder hợp lệ của một BST (mục 12D).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Chèn lần lượt | `O(n·h)` | `O(h)` | oracle, `≤ O(n²)` |
| 2. Tách mảng | `O(n²)` | `O(n)` | rõ ý nhất, cắt mảng |
| 3. Cận trên | `O(n)` | `O(h)` | **đáp án chính** |
| 4. Ngăn xếp | `O(n)` | `O(h)` | không đệ quy |
| 5. Tách bằng nhị phân | `O(n log n)` | `O(h)` | không cắt mảng |

Đo thực tế (`n = 100`; cây lệch và cây cân bằng):

In [18]:
def bst_random_preorder(rng, n, lo=1, hi=1000):
    """Preorder của một BST ngẫu nhiên n nút (giá trị khác nhau trong [lo, hi])."""
    vals = rng.sample(range(lo, hi + 1), n)
    root = TreeNode(vals[0])
    for v in vals[1:]:
        cur = root
        while True:
            if v < cur.val:
                if cur.left is None:
                    cur.left = TreeNode(v)
                    break
                cur = cur.left
            else:
                if cur.right is None:
                    cur.right = TreeNode(v)
                    break
                cur = cur.right
    return preorder_of(root)


def bench(fn, arg, repeat=300):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(arg)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(1008)
shapes = {"ngẫu nhiên": bst_random_preorder(rng, 100), "lệch phải (tăng)": list(range(1, 101)), "lệch trái (giảm)": list(range(100, 0, -1))}
for sname, pre in shapes.items():
    print(f"{sname}:")
    for name, fn in [("insert", bst_insert), ("split", bst_split), ("bound", bst_bound), ("stack", bst_stack), ("bisect", bst_bisect)]:
        print(f"    {name:7s}: {bench(fn, pre) * 1e6:8.1f} us")

ngẫu nhiên:
    insert :     44.0 us


    split  :     56.7 us


    bound  :     43.0 us


    stack  :     37.6 us


    bisect :     43.3 us
lệch phải (tăng):


    insert :    162.5 us


    split  :     37.2 us


    bound  :     28.2 us
    stack  :     22.2 us
    bisect :     56.2 us
lệch trái (giảm):


    insert :    176.1 us


    split  :    296.7 us
    bound  :     25.0 us
    stack  :     18.7 us
    bisect :     52.3 us


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi BST `n ≤ 8` và mọi hoán vị, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn `n = 100`.

Vì đáp án **duy nhất**, so sánh trực tiếp cây (qua `tup`) **và** kiểm tra tính hợp lệ `valid_answer`.

### A. Ví dụ của đề

In [19]:
IMPLS = {
    "insert": bst_insert,
    "split": bst_split,
    "bound": bst_bound,
    "stack": bst_stack,
    "bisect": bst_bisect,
    "Solution": lambda p: Solution().bstFromPreorder(list(p)),
    "SolutionInsert": lambda p: SolutionInsert().bstFromPreorder(list(p)),
    "SolutionSplit": lambda p: SolutionSplit().bstFromPreorder(list(p)),
    "SolutionBound": lambda p: SolutionBound().bstFromPreorder(list(p)),
    "SolutionStack": lambda p: SolutionStack().bstFromPreorder(list(p)),
    "SolutionBisect": lambda p: SolutionBisect().bstFromPreorder(list(p)),
}


def check(pre, want_tuple=None):
    pre = list(pre)
    snapshot = pre[:]
    results = {}
    for name in IMPLS:
        root = IMPLS[name](pre)
        assert valid_answer(root, pre), (name, pre)
        assert size(root) == len(pre)
        results[name] = tup(root)
    assert pre == snapshot
    first = next(iter(results.values()))
    assert all(v == first for v in results.values()), (pre, {k: v for k, v in results.items() if v != first})
    if want_tuple is not None:
        assert first == want_tuple, (pre, first, want_tuple)
    return first


for pre, want in ex:
    check(pre, tup(build_tree(want)))
    assert to_level(Solution().bstFromPreorder(pre)) == want
print("ví dụ đề: các cài đặt trả cùng một cây, đúng định dạng LeetCode và qua hàm kiểm tra hợp lệ")

ví dụ đề: các cài đặt trả cùng một cây, đúng định dạng LeetCode và qua hàm kiểm tra hợp lệ


### B. Edge case

- Một nút: `[5]`.
- Hai nút: `[1, 3]` (phải), `[3, 1]` (trái).
- Toàn bộ chỉ có con trái (giảm dần) / chỉ có con phải (tăng dần) — cây lệch dài 100.
- Cây đầy đủ `[4, 2, 1, 3, 6, 5, 7]`.
- Giá trị nhỏ nhất `1` và lớn nhất `1000`.
- Zigzag (trái rồi phải xen kẽ): `[5, 1, 4, 2, 3]`.
- Cây có cây con phải của cây con trái sâu (`[8, 5, 1, 7, 10, 12]`).

In [20]:
edge = [
    [5], [1, 3], [3, 1], [1000], [1], [4, 2, 1, 3, 6, 5, 7], [5, 1, 4, 2, 3], [8, 5, 1, 7, 10, 12], [2, 1, 3], [1, 2, 3, 4, 5], [5, 4, 3, 2, 1],
    [10, 5, 1, 7, 40, 50], [3, 2, 1, 6, 5, 4, 7], [50, 30, 20, 40, 70, 60, 80], [1, 1000], [1000, 1], [500, 1, 250, 1000],
]
for pre in edge:
    check(pre)
check(list(range(1, 101)))
check(list(range(100, 0, -1)))
print(len(edge) + 2, "edge case: mọi cài đặt cho cùng cây hợp lệ")
print("[5,1,4,2,3] ->", to_level(Solution().bstFromPreorder([5, 1, 4, 2, 3])))

19 edge case: mọi cài đặt cho cùng cây hợp lệ
[5,1,4,2,3] -> [5, 1, None, None, 4, 2, None, None, 3]


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [21]:
def bug_compare_with_prev(preorder):                   # so mỗi giá trị chỉ với nút ngay trước (bỏ qua tổ tiên)
    root = TreeNode(preorder[0])
    prev = root
    for v in preorder[1:]:
        node = TreeNode(v)
        if v < prev.val:
            prev.left = node
        else:
            prev.right = node
        prev = node
    return root


def bug_no_upper_bound(preorder):                      # đệ quy chỉ với cận dưới/không cận trên: phải tiêu thụ hết
    idx = 0
    n = len(preorder)

    def build(bound):
        nonlocal idx
        if idx == n:
            return None
        node = TreeNode(preorder[idx])
        idx += 1
        node.left = build(node.val) if idx < n and preorder[idx] < node.val else None
        node.right = build(bound) if idx < n else None          # BUG: không kiểm tra preorder[idx] > bound
        return node

    return build(float("inf"))


def bug_local_index(preorder):                         # chỉ số cục bộ thay vì dùng chung
    def build(i, bound):
        if i >= len(preorder) or preorder[i] > bound:
            return None, i
        node = TreeNode(preorder[i])
        node.left, j = build(i + 1, node.val)
        node.right, _ = build(i + 1, bound)             # BUG: dùng i + 1 thay vì j
        return node, j

    return build(0, float("inf"))[0]


def bug_split_off_by_one(preorder):                    # tách lệch: bỏ mất phần tử đầu khối phải
    if not preorder:
        return None
    k = 1
    while k < len(preorder) and preorder[k] < preorder[0]:
        k += 1
    return TreeNode(preorder[0], bug_split_off_by_one(preorder[1:k]), bug_split_off_by_one(preorder[k + 1:]))


def bug_stack_pop_once(preorder):                      # chỉ pop một lần
    root = TreeNode(preorder[0])
    stack = [root]
    for v in preorder[1:]:
        node = TreeNode(v)
        if stack[-1].val < v:
            last = stack.pop()
            last.right = node
        else:
            stack[-1].left = node
        stack.append(node)
    return root


def bug_balanced_from_sorted(preorder):                # dựng BST cân bằng từ mảng đã sắp (đúng BST, sai preorder)
    s = sorted(preorder)

    def build(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        return TreeNode(s[mid], build(lo, mid - 1), build(mid + 1, hi))

    return build(0, len(s) - 1)


def bug_sorted_insert(preorder):                       # chèn theo thứ tự đã sắp (chuỗi)
    return bst_insert(sorted(preorder))


def bug_mirror(preorder):                              # đổi nhầm: lớn hơn sang trái
    root = TreeNode(preorder[0])
    for v in preorder[1:]:
        cur = root
        while True:
            if v > cur.val:
                if cur.left is None:
                    cur.left = TreeNode(v)
                    break
                cur = cur.left
            else:
                if cur.right is None:
                    cur.right = TreeNode(v)
                    break
                cur = cur.right
    return root


BUGS = [bug_compare_with_prev, bug_no_upper_bound, bug_local_index, bug_split_off_by_one, bug_stack_pop_once, bug_balanced_from_sorted, bug_sorted_insert, bug_mirror]
probe = [[8, 5, 1, 7, 10, 12], [1, 3], [3, 1], [2, 1, 3], [5, 1, 4, 2, 3], [4, 2, 1, 3, 6, 5, 7], [3, 2, 1], [1, 2, 3], [5, 2, 3, 1, 7]]
for bug in BUGS:
    found = None
    for pre in probe:
        try:
            got = tup(bug(pre))
        except Exception as e:
            got = type(e).__name__
        if got != tup(bst_insert(pre)):
            found = (pre, got)
            break
    assert found, bug.__name__
    print(f"{bug.__name__:26s} sai trên {found[0]}: ra {found[1]}")

bug_compare_with_prev      sai trên [8, 5, 1, 7, 10, 12]: ra (8, (5, (1, None, (7, None, (10, None, (12, None, None)))), None), None)
bug_no_upper_bound         sai trên [8, 5, 1, 7, 10, 12]: ra (8, (5, (1, None, (7, None, (10, None, (12, None, None)))), None), None)
bug_local_index            sai trên [8, 5, 1, 7, 10, 12]: ra (8, (5, (1, None, None), (1, None, (7, None, None))), (5, (1, None, None), (1, None, (7, None, (10, None, (12, None, None))))))
bug_split_off_by_one       sai trên [8, 5, 1, 7, 10, 12]: ra (8, (5, (1, None, None), None), (12, None, None))
bug_stack_pop_once         sai trên [8, 5, 1, 7, 10, 12]: ra (8, (5, (1, None, (7, None, (10, None, (12, None, None)))), None), None)
bug_balanced_from_sorted   sai trên [8, 5, 1, 7, 10, 12]: ra (7, (1, None, (5, None, None)), (10, (8, None, None), (12, None, None)))
bug_sorted_insert          sai trên [8, 5, 1, 7, 10, 12]: ra (1, None, (5, None, (7, None, (8, None, (10, None, (12, None, None))))))
bug_mirror                 sai

### D. Vét cạn mọi BST `n ≤ 8` và mọi hoán vị

1. Với mỗi `n = 1..8`: sinh **mọi BST** `n` nút trên nhãn `1..n` bằng cách chọn gốc `r`, ghép mọi cây con trái trên `1..r−1` với mọi cây con phải trên `r+1..n` (`Catalan(n)` cây). Lấy preorder của từng cây; các preorder phải **đôi một khác nhau** (đơn ánh); mọi cách dựng lại phải trả đúng cây gốc.
2. Với mỗi `n ≤ 7`: xét mọi `n!` hoán vị của `1..n`; hoán vị là preorder hợp lệ của BST ⇔ kiểm tra bằng **vị từ độc lập** `valid_preorder_def` (đệ quy theo định nghĩa: sau gốc là khối nhỏ hơn rồi khối lớn hơn, đệ quy); số hoán vị hợp lệ phải bằng `Catalan(n)`; với mỗi hoán vị hợp lệ mọi cài đặt cho BST có preorder đúng bằng hoán vị.

In [22]:
def all_bsts(lo, hi):
    """Mọi BST (dạng tuple) trên các nhãn lo..hi."""
    if lo > hi:
        return [None]
    out = []
    for r in range(lo, hi + 1):
        for lt in all_bsts(lo, r - 1):
            for rt in all_bsts(r + 1, hi):
                out.append((r, lt, rt))
    return out


def preorder_tuple(t, acc=None):
    acc = [] if acc is None else acc
    if t is not None:
        acc.append(t[0])
        preorder_tuple(t[1], acc)
        preorder_tuple(t[2], acc)
    return acc


def catalan(n):
    c = 1
    for i in range(n):
        c = c * 2 * (2 * i + 1) // (i + 2)
    return c


t0 = time.perf_counter()
total = 0
for n in range(1, 9):
    trees = all_bsts(1, n)
    assert len(trees) == catalan(n)
    pres = set()
    for t in trees:
        pre = preorder_tuple(t)
        assert tuple(pre) not in pres                      # đơn ánh: hai BST khác nhau có preorder khác nhau
        pres.add(tuple(pre))
        for name in IMPLS:
            assert tup(IMPLS[name](pre)) == t, (name, pre)
    total += len(trees)
    print(f"  n = {n}: {len(trees)} BST (= Catalan({n})), {len(pres)} preorder phân biệt")
print(total, "BST: mọi cài đặt dựng lại đúng cây gốc từ preorder;", round(time.perf_counter() - t0, 1), "s")

  n = 1: 1 BST (= Catalan(1)), 1 preorder phân biệt
  n = 2: 2 BST (= Catalan(2)), 2 preorder phân biệt
  n = 3: 5 BST (= Catalan(3)), 5 preorder phân biệt
  n = 4: 14 BST (= Catalan(4)), 14 preorder phân biệt
  n = 5: 42 BST (= Catalan(5)), 42 preorder phân biệt
  n = 6: 132 BST (= Catalan(6)), 132 preorder phân biệt
  n = 7: 429 BST (= Catalan(7)), 429 preorder phân biệt


  n = 8: 1430 BST (= Catalan(8)), 1430 preorder phân biệt
2055 BST: mọi cài đặt dựng lại đúng cây gốc từ preorder; 0.1 s


In [23]:
def valid_preorder_def(seq):
    """Vị từ độc lập (theo định nghĩa): seq là preorder của một BST <=> sau gốc là khối < gốc rồi khối > gốc, và mỗi khối đệ quy hợp lệ."""
    if len(seq) <= 1:
        return True
    root = seq[0]
    k = 1
    while k < len(seq) and seq[k] < root:
        k += 1
    if any(x < root for x in seq[k:]):
        return False
    return valid_preorder_def(seq[1:k]) and valid_preorder_def(seq[k:])


t0 = time.perf_counter()
for n in range(1, 8):
    valid = 0
    for perm in itertools.permutations(range(1, n + 1)):
        if valid_preorder_def(perm):
            valid += 1
            for name in IMPLS:
                root = IMPLS[name](list(perm))
                assert preorder_of(root) == list(perm) and is_bst(root), (name, perm)
    assert valid == catalan(n), (n, valid)
    print(f"  n = {n}: {valid} / {math.factorial(n)} hoán vị là preorder BST hợp lệ (= Catalan({n}))")
print("số hoán vị hợp lệ = Catalan(n) với mọi n ≤ 7; mọi cài đặt đúng;", round(time.perf_counter() - t0, 1), "s")

  n = 1: 1 / 1 hoán vị là preorder BST hợp lệ (= Catalan(1))
  n = 2: 2 / 2 hoán vị là preorder BST hợp lệ (= Catalan(2))
  n = 3: 5 / 6 hoán vị là preorder BST hợp lệ (= Catalan(3))
  n = 4: 14 / 24 hoán vị là preorder BST hợp lệ (= Catalan(4))
  n = 5: 42 / 120 hoán vị là preorder BST hợp lệ (= Catalan(5))
  n = 6: 132 / 720 hoán vị là preorder BST hợp lệ (= Catalan(6))
  n = 7: 429 / 5040 hoán vị là preorder BST hợp lệ (= Catalan(7))
số hoán vị hợp lệ = Catalan(n) với mọi n ≤ 7; mọi cài đặt đúng; 0.1 s


### E. Tính chất / ngẫu nhiên (seed cố định)

Với preorder của BST ngẫu nhiên (giá trị tới `1000`, `n` tới `100`, nhiều hình dạng): (1) mọi cách trả **cùng** một cây và qua `valid_answer`; (2) `inorder == sorted(preorder)`; (3) `to_level` rồi `build_tree` rồi `preorder_of` cho lại preorder (khứ hồi kiểu LeetCode); (4) cộng một hằng số vào mọi giá trị không đổi **hình dạng**; (5) chiều cao ≤ `n`, số nút bằng `n`; (6) không sửa đầu vào.

In [24]:
def height(root):
    best, stack = 0, [(root, 1)]
    while stack:
        x, d = stack.pop()
        if x:
            best = max(best, d)
            stack.append((x.left, d + 1))
            stack.append((x.right, d + 1))
    return best


def shape(t):
    return None if t is None else (shape(t[1]), shape(t[2]))


rng = random.Random(20240108)
t0 = time.perf_counter()
heights = []
for it in range(2000):
    n = rng.choice([1, 2, 3, 5, 10, 30, 60, 100])
    if rng.random() < 0.15:
        pre = sorted(rng.sample(range(1, 1001), n), reverse=rng.random() < 0.5)       # cây lệch
    else:
        pre = bst_random_preorder(rng, n)
    t = check(pre)                                                                    # (1) (6)
    root = from_tuple(t)
    assert inorder_of(root) == sorted(pre)                                            # (2)
    assert preorder_of(build_tree(to_level(root))) == pre                             # (3)
    pre2 = [v + 7 for v in pre]
    if max(pre2) <= 1007:
        assert shape(tup(Solution().bstFromPreorder(pre2))) == shape(t)               # (4)
    assert size(root) == n and height(root) <= n                                      # (5)
    heights.append(height(root))
print("2000 preorder ngẫu nhiên (n tới 100): 6 tính chất đúng; chiều cao lớn nhất gặp:", max(heights), ";", round(time.perf_counter() - t0, 1), "s")

2000 preorder ngẫu nhiên (n tới 100): 6 tính chất đúng; chiều cao lớn nhất gặp: 100 ; 0.9 s


### F. Giới hạn

`n = 100`: cây lệch phải (`1..100`), lệch trái (`100..1`), zigzag (xen kẽ trái/phải sâu 100), cây cân bằng, giá trị tới `1000`.

In [25]:
def zigzag_preorder(n):
    """Preorder của cây mà mỗi nút có đúng một con, xen kẽ trái/phải."""
    lo, hi = 1, n
    out = []
    left_turn = True
    for _ in range(n):
        if left_turn:
            out.append(hi)           # nút hiện tại lớn nhất khoảng còn lại; con tiếp theo ở bên trái
            hi -= 1
        else:
            out.append(lo)           # nút nhỏ nhất khoảng còn lại; con tiếp theo ở bên phải
            lo += 1
        left_turn = not left_turn
    return out


def balanced_preorder(values):
    values = sorted(values)

    def go(lo, hi):
        if lo > hi:
            return []
        mid = (lo + hi) // 2
        return [values[mid]] + go(lo, mid - 1) + go(mid + 1, hi)

    return go(0, len(values) - 1)


t0 = time.perf_counter()
cases = {
    "lệch phải": list(range(1, 101)), "lệch trái": list(range(100, 0, -1)), "zigzag": zigzag_preorder(100),
    "cân bằng": balanced_preorder(range(1, 101)), "giá trị lớn": balanced_preorder(range(901, 1001)),
}
for name, pre in cases.items():
    t = check(pre)
    print(f"  {name:12s}: chiều cao {height(from_tuple(t)):3d}")
assert height(from_tuple(check(zigzag_preorder(100)))) == 100
assert height(from_tuple(check(list(range(1, 101))))) == 100
print("n = 100, 5 hình dạng: mọi cài đặt đúng;", round(time.perf_counter() - t0, 2), "s")

  lệch phải   : chiều cao 100
  lệch trái   : chiều cao 100
  zigzag      : chiều cao 100
  cân bằng    : chiều cao   7
  giá trị lớn : chiều cao   7
n = 100, 5 hình dạng: mọi cài đặt đúng; 0.01 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: so sánh mỗi giá trị chỉ với nút ngay trước

In [26]:
pre = [8, 5, 1, 7, 10, 12]
print("bug:", to_level(bug_compare_with_prev(pre)), "| đúng:", to_level(Solution().bstFromPreorder(pre)))
assert not valid_answer(bug_compare_with_prev(pre), pre)

bug: [8, 5, None, 1, None, None, 7, None, 10, None, 12] | đúng: [8, 5, 10, 1, 7, None, 12]


**Vì sao sai:** `10 > 7` nhưng `10` phải là con phải của `8` (tổ tiên xa), không phải con phải của `7`. Phải xét **tổ tiên** (cận trên hoặc ngăn xếp).

### Lỗi 2: thiếu cận trên khi đệ quy

In [27]:
pre = [8, 5, 1, 7, 10, 12]
print("bug:", to_level(bug_no_upper_bound(pre)), "| đúng:", to_level(Solution().bstFromPreorder(pre)))
assert not valid_answer(bug_no_upper_bound(pre), pre)

bug: [8, 5, None, 1, None, None, 7, None, 10, None, 12] | đúng: [8, 5, 10, 1, 7, None, 12]


**Vì sao sai:** giá trị lớn hơn cận (`10 > 8`) phải **kết thúc** cây con hiện tại, không được gắn vào dưới nút cuối. **Đúng:** `if preorder[idx] > bound: return None`.

### Lỗi 3: con trỏ chỉ số cục bộ

In [28]:
pre = [2, 1, 3]
print("bug:", to_level(bug_local_index(pre)), "| đúng:", to_level(Solution().bstFromPreorder(pre)))
bad = next(p for p in probe if tup(bug_local_index(p)) != tup(bst_insert(p)))
print("phản ví dụ:", bad, "-> bug", to_level(bug_local_index(bad)), "| đúng", to_level(bst_insert(bad)))

bug: [2, 1, 1, None, None, None, 3] | đúng: [2, 1, 3]
phản ví dụ: [8, 5, 1, 7, 10, 12] -> bug [8, 5, 5, 1, 1, 1, 1, None, None, None, 7, None, None, None, 7, None, None, None, 10, None, 12] | đúng [8, 5, 10, 1, 7, None, 12]


**Vì sao sai:** sau khi dựng xong cây con trái, chỉ số phải **tiến** qua toàn bộ các giá trị đã dùng; truyền `i + 1` cho nhánh phải làm đọc lại giá trị cũ. **Đúng:** `nonlocal idx` (dùng chung) hoặc trả về chỉ số mới.

### Lỗi 4: tách lệch một phần tử

In [29]:
pre = [8, 5, 1, 7, 10, 12]
print("bug:", to_level(bug_split_off_by_one(pre)), "| đúng:", to_level(Solution().bstFromPreorder(pre)))
assert preorder_of(bug_split_off_by_one(pre)) != pre

bug: [8, 5, 12, 1] | đúng: [8, 5, 10, 1, 7, None, 12]


**Vì sao sai:** khối phải bắt đầu **ngay tại `k`** (`pre[k:]`), không phải `k + 1`; nếu không phần tử `pre[k]` bị mất.

### Lỗi 5: ngăn xếp chỉ `pop` một lần

In [30]:
pre = [8, 5, 1, 7, 10, 12]
print("bug:", to_level(bug_stack_pop_once(pre)), "| đúng:", to_level(Solution().bstFromPreorder(pre)))
assert not valid_answer(bug_stack_pop_once(pre), pre)

bug: [8, 5, None, 1, None, None, 7, None, 10, None, 12] | đúng: [8, 5, 10, 1, 7, None, 12]


**Vì sao sai:** khi `v` lớn hơn nhiều nút trên ngăn xếp (như `10` lớn hơn `7`, `8`), phải pop **hết** các nút nhỏ hơn để tìm cha thật sự (`8`). **Đúng:** `while stack and stack[-1].val < v: last = stack.pop()`.

### Lỗi 6 (bonus): dựng BST cân bằng từ mảng đã sắp

In [31]:
pre = [8, 5, 1, 7, 10, 12]
t = bug_balanced_from_sorted(pre)
print("bug:", to_level(t), "| là BST?", is_bst(t), "| preorder:", preorder_of(t), "| đúng preorder:", pre)
assert is_bst(t) and preorder_of(t) != pre

bug: [7, 1, 10, None, 5, 8, 12] | là BST? True | preorder: [7, 1, 5, 10, 8, 12] | đúng preorder: [8, 5, 1, 7, 10, 12]


**Vì sao sai:** có **nhiều** BST cùng tập giá trị; đề đòi cây có **preorder đúng bằng đầu vào**. Chỉ "là BST" chưa đủ.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "dựng cây từ một phép duyệt, kèm tính chất sắp thứ tự (BST)" → preorder BST không cần trung thứ tự: dùng **cận** (giá trị chạy tới đâu thuộc cây con nào) hoặc **ngăn xếp đơn điệu**.

**Mẹo:**

1. Preorder BST = gốc, khối nhỏ hơn, khối lớn hơn.
2. Đệ quy với cận trên, một con trỏ dùng chung ⇒ `O(n)`.
3. Ngăn xếp: pop hết các nút nhỏ hơn; nút pop cuối là cha (con phải).
4. Điểm chia có thể tìm bằng nhị phân vì vị từ đơn điệu.
5. Với preorder bất kỳ (không phải BST) cần thêm inorder.
6. Oracle: chèn lần lượt; hàm hợp lệ: BST + preorder khớp.

**Họ bài cùng pattern:** Construct Binary Tree from Preorder and Inorder Traversal (105), Verify Preorder Sequence in BST (255, Premium), Serialize and Deserialize BST (449), Convert Sorted Array to BST (108), Validate BST (98), Recover BST from Preorder (1028).

## 15. Biến thể

1. **BST từ postorder** (đọc ngược: gốc, phải, trái với cận dưới).
2. **BST từ thứ tự theo tầng** (chèn lần lượt).
3. **Đếm số preorder hợp lệ** của BST `n` nút (= Catalan).

Cài đặt và đối chiếu vét cạn:

In [32]:
def bst_from_postorder(post):
    idx = len(post) - 1

    def build(bound):                                    # giá trị > bound thuộc cây con này
        nonlocal idx
        if idx < 0 or post[idx] < bound:
            return None
        node = TreeNode(post[idx])
        idx -= 1
        node.right = build(node.val)                     # đọc ngược: gốc, phải, trái
        node.left = build(bound)
        return node

    return build(float("-inf"))


def postorder_of(root):
    out = []
    stack = [(root, False)]
    while stack:
        x, done = stack.pop()
        if x is None:
            continue
        if done:
            out.append(x.val)
        else:
            stack.append((x, True))
            stack.append((x.right, False))
            stack.append((x.left, False))
    return out


def bst_from_levelorder(level):
    return bst_insert(level)


def levelorder_of(root):
    out, queue = [], deque([root])
    while queue:
        x = queue.popleft()
        if x:
            out.append(x.val)
            queue.append(x.left)
            queue.append(x.right)
    return out


for n in range(1, 8):
    for t in all_bsts(1, n):
        root = from_tuple(t)
        assert tup(bst_from_postorder(postorder_of(root))) == t               # (1)
        assert tup(bst_from_levelorder(levelorder_of(root))) == t             # (2)
for n in range(1, 8):
    cnt = len({tuple(preorder_tuple(t)) for t in all_bsts(1, n)})
    assert cnt == catalan(n)                                                  # (3)
print("BST từ postorder / từ theo tầng / số preorder = Catalan: khớp vét cạn (n ≤ 7)")

BST từ postorder / từ theo tầng / số preorder = Catalan: khớp vét cạn (n ≤ 7)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chèn lần lượt | theo định nghĩa BST | `O(n·h)` | `O(h)` | oracle |
| 2. Tách mảng | khối nhỏ / khối lớn | `O(n²)` | `O(n)` | dễ hiểu |
| 3. Cận trên | một con trỏ chung | `O(n)` | `O(h)` | **đáp án chính** |
| 4. Ngăn xếp | pop nút nhỏ hơn | `O(n)` | `O(h)` | không đệ quy |
| 5. Tách nhị phân | vị từ đơn điệu | `O(n log n)` | `O(h)` | không cắt mảng |

Mọi cách dựng lại đúng cây gốc từ preorder trên toàn bộ BST `n ≤ 8` (mục 12D).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy (in dạng `to_level`):

In [33]:
print(to_level(Solution().bstFromPreorder([4, 2, 6])), to_level(Solution().bstFromPreorder([3, 2, 1])), to_level(Solution().bstFromPreorder([1, 2, 3])),
      to_level(Solution().bstFromPreorder([5])), to_level(Solution().bstFromPreorder([5, 1, 4, 2, 3])))

[4, 2, 6] [3, 2, None, 1] [1, None, 2, None, 3] [5] [5, 1, None, None, 4, 2, None, None, 3]


### Bài 2 (tính tay)

`preorder = [10, 5, 1, 7, 40, 50]`. (a) Khối trái và khối phải của gốc `10`. (b) Dựng cây và viết dạng `[...]` theo tầng. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện đệ quy với cận trên:

In [34]:
def bound_todo(preorder):
    idx, n = 0, len(preorder)

    def build(bound):
        nonlocal idx
        if idx == n:                    # TODO 1: thêm điều kiện: preorder[idx] > bound
            return None
        node = None                     # TODO 2: tạo TreeNode(preorder[idx]) rồi idx += 1
        # TODO 3: node.left = build(node.val); node.right = build(bound)
        return node

    return build(float("inf"))


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(to_level(bound_todo(p)) == w for p, w in ex)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **ngăn xếp**:

In [35]:
class MySolution:
    def bstFromPreorder(self, preorder):
        pass            # TODO


# Khi làm xong:
# assert all(to_level(MySolution().bstFromPreorder(p)) == w for p, w in ex)

### Bài 5 (mini challenge): BST từ postorder

Viết `from_postorder(post)` dựng BST từ duyệt **hậu thứ tự** (trái, phải, nút). Gợi ý: đọc mảng từ cuối (gốc, rồi cây con phải, rồi cây con trái) với **cận dưới**. Kiểm bằng `postorder_of` (mục 15).

In [36]:
def from_postorder(post):
    pass            # TODO


# Khi làm xong:
# assert postorder_of(from_postorder([1, 7, 5, 50, 40, 10])) == [1, 7, 5, 50, 40, 10]

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: gốc là phần tử đầu; các giá trị nhỏ hơn đi bên trái.
- Mức 2: `[3,2,1]` chỉ có con trái; `[1,2,3]` chỉ có con phải.
- Mức 3: `[4,2,6]`, `[3,2,null,1]`, `[1,null,2,null,3]`, `[5]`; với `[5,1,4,2,3]` gốc `5` chỉ có cây con trái `1(·,4(2(·,3),·))` — chạy mục 17 để thấy dạng theo tầng.

**Bài 2**

- Mức 1: gốc `10`; phần tử `> 10` đầu tiên là `40`.
- Mức 2: khối trái `[5, 1, 7]`, khối phải `[40, 50]`.
- Mức 3: `10(5(1,7), 40(·,50))` = `[10,5,40,1,7,null,50]`.

**Bài 3**

- Mức 1: `if idx == n or preorder[idx] > bound: return None`.
- Mức 2: `node = TreeNode(preorder[idx]); idx += 1`.
- Mức 3: `node.left = build(node.val); node.right = build(bound)`.

**Bài 4**

- Mức 1: `stack = [root]`.
- Mức 2: `while stack and stack[-1].val < v: last = stack.pop()`.
- Mức 3: `last.right = node` nếu có `last`, ngược lại `stack[-1].left = node`; rồi `stack.append(node)`.

**Bài 5**

- Mức 1: gốc là phần tử **cuối** của postorder.
- Mức 2: duyệt từ cuối: gốc, rồi cây con **phải**, rồi cây con trái.
- Mức 3: `build(bound)` dừng khi `post[idx] < bound`; phải dùng cận dưới `node.val`, trái dùng cận `bound`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [37]:
vals = (to_level(Solution().bstFromPreorder([4, 2, 6])), to_level(Solution().bstFromPreorder([3, 2, 1])), to_level(Solution().bstFromPreorder([1, 2, 3])),
        to_level(Solution().bstFromPreorder([5])), to_level(Solution().bstFromPreorder([5, 1, 4, 2, 3])))
print(vals)
assert vals[:4] == ([4, 2, 6], [3, 2, None, 1], [1, None, 2, None, 3], [5])
assert preorder_of(build_tree(vals[4])) == [5, 1, 4, 2, 3] and is_bst(build_tree(vals[4]))
print("Bài 1 OK")

([4, 2, 6], [3, 2, None, 1], [1, None, 2, None, 3], [5], [5, 1, None, None, 4, 2, None, None, 3])
Bài 1 OK


### Bài 2

In [38]:
pre = [10, 5, 1, 7, 40, 50]
k = next(i for i in range(1, len(pre)) if pre[i] > pre[0])
print("khối trái:", pre[1:k], "| khối phải:", pre[k:])
assert pre[1:k] == [5, 1, 7] and pre[k:] == [40, 50]
res = to_level(Solution().bstFromPreorder(pre))
assert res == [10, 5, 40, 1, 7, None, 50]
print("Bài 2 OK:", res)

khối trái: [5, 1, 7] | khối phải: [40, 50]
Bài 2 OK: [10, 5, 40, 1, 7, None, 50]


### Bài 3

In [39]:
def bound_answer(preorder):
    idx, n = 0, len(preorder)

    def build(bound):
        nonlocal idx
        if idx == n or preorder[idx] > bound:
            return None
        node = TreeNode(preorder[idx])
        idx += 1
        node.left = build(node.val)
        node.right = build(bound)
        return node

    return build(float("inf"))


assert all(to_level(bound_answer(p)) == w for p, w in ex)
for n in range(1, 8):
    for t in all_bsts(1, n):
        assert tup(bound_answer(preorder_tuple(t))) == t
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [40]:
class MySolutionAnswer:
    def bstFromPreorder(self, preorder):
        root = TreeNode(preorder[0])
        stack = [root]
        for v in preorder[1:]:
            node = TreeNode(v)
            last = None
            while stack and stack[-1].val < v:
                last = stack.pop()
            if last is not None:
                last.right = node
            else:
                stack[-1].left = node
            stack.append(node)
        return root


assert all(to_level(MySolutionAnswer().bstFromPreorder(p)) == w for p, w in ex)
for n in range(1, 8):
    for t in all_bsts(1, n):
        assert tup(MySolutionAnswer().bstFromPreorder(preorder_tuple(t))) == t
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [41]:
def from_postorder(post):
    idx = len(post) - 1

    def build(bound):
        nonlocal idx
        if idx < 0 or post[idx] < bound:
            return None
        node = TreeNode(post[idx])
        idx -= 1
        node.right = build(node.val)
        node.left = build(bound)
        return node

    return build(float("-inf"))


assert postorder_of(from_postorder([1, 7, 5, 50, 40, 10])) == [1, 7, 5, 50, 40, 10]
for n in range(1, 8):
    for t in all_bsts(1, n):
        root = from_tuple(t)
        assert tup(from_postorder(postorder_of(root))) == t
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (dựng BST từ preorder → đệ quy với cận trên, con trỏ dùng chung):**

```python
class Solution:
    def bstFromPreorder(self, preorder: List[int]) -> Optional[TreeNode]:
        idx, n = 0, len(preorder)

        def build(bound):
            nonlocal idx
            if idx == n or preorder[idx] > bound:
                return None
            node = TreeNode(preorder[idx])
            idx += 1
            node.left = build(node.val)
            node.right = build(bound)
            return node

        return build(float("inf"))
```

**Checklist trước khi submit:**

- [ ] Con trỏ `idx` **dùng chung** (`nonlocal`), không truyền theo giá trị?
- [ ] Dừng khi `idx == n` hoặc `preorder[idx] > bound`?
- [ ] Cây trái dùng cận `node.val`; cây phải dùng cận của **tổ tiên** (`bound`)?
- [ ] Nếu dùng ngăn xếp: pop **hết** các nút nhỏ hơn `v`, nút pop cuối là cha (con phải)?
- [ ] Đã thử `[8,5,1,7,10,12]`, `[1,3]`, cây lệch 100 nút, một nút?
- [ ] Đáp án duy nhất vì giá trị khác nhau (chỉ cần "là BST" chưa đủ: preorder phải khớp)?
- [ ] Đúng tên `class Solution`, `bstFromPreorder(self, preorder)`, không in debug.